#### Load env variables

In [ ]:
%pip install anthropic python-dotenv
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper Function
def store_user_message(allMessages, text):
    user_message= {"role":"user", "content":text}
    allMessages.append(user_message)
    
def store_assistant_message(allMessages, text):
    assistant_message= {"role":"assistant", "content":text}
    allMessages.append(assistant_message)  
    
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = dict(
        model=model,
        max_tokens=1000,
        messages=messages,
        temperature=temperature
    )
    
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    message = client.messages.create(**params)
    return message.content[0].text


In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    allMessages = []
    
    store_user_message(allMessages, prompt);
    store_assistant_message(allMessages,"```json")
    
    text = chat(allMessages, stop_sequences=["```"])
    
    return json.loads(text)

In [ ]:
dataset = generate_dataset()

with open("dataset.json","w") as f:
    json.dump(dataset, f, indent=2)

In [15]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}
    """
    
    messages = []
    store_user_message(messages, prompt)
    text = chat(messages)
    return text
    

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grading - TODO
    score = 10
    
    return {
        "output" : output,
        "test_case" : test_case,
        "score" : score
    }

def run_eval(dataset):
    """Loads the dataset and calls  run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    return results

In [16]:
with open("dataset.json","r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri):\n    \"\"\"\n    Parse and return the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not isinstance(s3_uri, str):\n        raise ValueError(\"S3 URI must be a string\")\n    \n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"S3 URI must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Bucket name cannot be empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"